In [1]:
# prepare data
import pickle 
import numpy as np
with open("/data/transformer_project/transforemer_model/train_data/training/cr/temp/ribo_centers_new.pkl", "rb") as f:
    ribo_centers = pickle.load(f)

In [2]:
for i in ribo_centers:
    array = np.array(ribo_centers[i]) 
    array = array[:, [2, 1, 0]]
    np.savetxt(f"/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/raw_data/{i}.txt", array, delimiter="\t")

In [6]:
from pathlib import Path
import json
import csv
import subprocess
import time
from datetime import datetime

# Two-fold split provided by user
fold1 = [
    "24092009",
    "24092014",
    "24092017",
    "24092023",
    "24092025",
    "24092029",
    "24092031",
]
fold2 = [
    "24092002",
    "24092013",
    "24092016",
    "24092020",
    "24092024",
    "24092026",
    "24092030",
]

repo = Path("/home/feity/compare/deepETPicker/DeepETPicker")
base = Path("/data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome")
raw = base / "raw_data"
coords_dir = raw / "coords"
cfg_dir = base / "configs"
cfg_dir.mkdir(parents=True, exist_ok=True)
runner_python = Path("/data/biosoftware/miniconda3/miniconda3/envs/deepET/bin/python")

# Discover and validate dataset files
all_tomos = sorted(p.stem for p in raw.glob("*.mrc"))
all_coords = sorted(p.stem for p in coords_dir.glob("*.coords"))
print("Tomograms:", all_tomos)
print("Coords:", all_coords)
assert all_tomos == all_coords, "Tomogram and coord file stems do not match"
assert len(all_tomos) == 14, f"Expected 14 tomograms, found {len(all_tomos)}"

# Validate fold assignment covers all 14 exactly once
all_ids_from_folds = sorted(fold1 + fold2)
assert set(fold1).isdisjoint(set(fold2)), "fold1 and fold2 overlap"
assert all_ids_from_folds == all_tomos, "Fold lists must match the 14 tomogram IDs in raw_data"

# Convert tomo names to integer IDs expected by generate_train_config.py
idx_to_tomo = {i: t for i, t in enumerate(all_tomos)}
tomo_to_idx = {t: i for i, t in idx_to_tomo.items()}

print("\nTomogram index mapping:")
for i, t in idx_to_tomo.items():
    print(f"  {i}: {t}")

# Ribosome-specific parameters (larger particle than hsp60)
LABEL_DIAMETER = 25
OCP_DIAMETER = "37"
BATCH_SIZE = "8"
BLOCK_SIZE = "72"
PAD_SIZE = "12"
LEARNING_RATE = "0.001"
MAX_EPOCH = "24"
THRESHOLD = "0.5"

# Preprocess config (raw data already prepared)
pre_config = {
    "dset_name": "ribosome_preprocess",
    "base_path": str(base),
    "coord_path": str(coords_dir),
    "coord_format": ".coords",
    "tomo_path": str(raw),
    "tomo_format": ".mrc",
    "num_cls": 1,
    "label_type": "gaussian",
    "label_diameter": LABEL_DIAMETER,
    "ocp_type": "sphere",
    "ocp_diameter": OCP_DIAMETER,
    "norm_type": "standardization",
}

pre_cfg_path = base / "ribosome_preprocess.py"
pre_cfg_path.write_text("pre_config=" + json.dumps(pre_config, indent=12))
print(f"\nSaved preprocessing config to {pre_cfg_path}")

# Two CV folds: train on one set, validate/test on the other
fold_specs = [
    {
        "name": "fold1_train_fold2_test",
        "train_tomos": fold1,
        "val_tomos": fold2,
        "gpu_id": "0",
    },
    {
        "name": "fold2_train_fold1_test",
        "train_tomos": fold2,
        "val_tomos": fold1,
        "gpu_id": "0",
    },
]


def run_cmd_capture(cmd, log_name):
    log_path = base / log_name
    start = time.time()
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo / "bin",
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )
    duration_sec = round(time.time() - start, 2)
    tail = log_path.read_text().splitlines()[-20:]
    print(f"[{log_name}] exit={proc.returncode}, duration={duration_sec}s")
    if tail:
        print("\n".join(tail))
    return proc.returncode, log_path, duration_sec


# 1) Preprocess once
pre_code, pre_log, pre_sec = run_cmd_capture(
    [runner_python, str(repo / "bin" / "preprocess.py"), "--pre_configs", str(pre_cfg_path)],
    "ribosome_preprocess.log",
)
if pre_code != 0:
    raise RuntimeError(f"Preprocess failed. Check log: {pre_log}")

# 2) Run 2-fold training + testing
results = []
for spec in fold_specs:
    split_name = spec["name"]
    train_ids = ",".join(str(tomo_to_idx[t]) for t in spec["train_tomos"])
    val_ids = ",".join(str(tomo_to_idx[t]) for t in spec["val_tomos"])
    dset_name = f"ribosome_{split_name}"
    train_cfg_path = cfg_dir / f"{dset_name}.py"

    print("\n" + "=" * 100)
    print(f"Split: {split_name}")
    print(f"Train tomos: {spec['train_tomos']}")
    print(f"Val/Test tomos: {spec['val_tomos']}")
    print(f"train_ids={train_ids}")
    print(f"val_ids={val_ids}")

    gen_cmd = [
        runner_python,
        str(repo / "bin" / "generate_train_config.py"),
        "--pre_configs",
        str(pre_cfg_path),
        "--dset_name",
        dset_name,
        "--cfg_save_path",
        str(cfg_dir),
        "--train_set_ids",
        train_ids,
        "--val_set_ids",
        val_ids,
        "--batch_size",
        BATCH_SIZE,
        "--block_size",
        BLOCK_SIZE,
        "--pad_size",
        PAD_SIZE,
        "--learning_rate",
        LEARNING_RATE,
        "--max_epoch",
        MAX_EPOCH,
        "--threshold",
        THRESHOLD,
        "--gpu_id",
        spec["gpu_id"],
    ]
    gen_code, gen_log, gen_sec = run_cmd_capture(gen_cmd, f"{dset_name}_generate.log")

    row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "split": split_name,
        "train_tomos": spec["train_tomos"],
        "val_tomos": spec["val_tomos"],
        "train_ids": train_ids,
        "val_ids": val_ids,
        "dset_name": dset_name,
        "train_config": str(train_cfg_path),
        "generate_exit": gen_code,
        "generate_log": str(gen_log),
        "generate_seconds": gen_sec,
        "train_exit": None,
        "train_log": "",
        "train_seconds": None,
        "checkpoint": "",
        "test_exit": None,
        "test_log": "",
        "test_seconds": None,
        "status": "pending",
    }

    if gen_code != 0:
        row["status"] = "generate_failed"
        results.append(row)
        continue

    train_cmd = [
        runner_python,
        str(repo / "bin" / "train_bash.py"),
        "--train_configs",
        str(train_cfg_path),
    ]
    train_code, train_log, train_sec = run_cmd_capture(train_cmd, f"{dset_name}_train.log")
    row["train_exit"] = train_code
    row["train_log"] = str(train_log)
    row["train_seconds"] = train_sec

    if train_code != 0:
        row["status"] = "train_failed"
        results.append(row)
        continue

    ckpt_candidates = sorted((base / "runs" / dset_name).glob("**/*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpt_candidates:
        row["status"] = "no_checkpoint"
        results.append(row)
        continue

    best_ckpt = ckpt_candidates[-1]
    row["checkpoint"] = str(best_ckpt)
    print(f"Using checkpoint: {best_ckpt}")

    test_cmd = [
        runner_python,
        str(repo / "bin" / "test_bash.py"),
        "--train_configs",
        str(train_cfg_path),
        "--checkpoints",
        str(best_ckpt),
    ]
    test_code, test_log, test_sec = run_cmd_capture(test_cmd, f"{dset_name}_test.log")
    row["test_exit"] = test_code
    row["test_log"] = str(test_log)
    row["test_seconds"] = test_sec
    row["status"] = "ok" if test_code == 0 else "test_failed"

    results.append(row)

# 3) Save split summary
results_json = base / "two_fold_results.json"
results_csv = base / "two_fold_results.csv"
results_json.write_text(json.dumps(results, indent=2))

with results_csv.open("w", newline="") as f:
    fieldnames = list(results[0].keys()) if results else [
        "timestamp", "split", "train_tomos", "val_tomos", "train_ids", "val_ids", "dset_name",
        "train_config", "generate_exit", "generate_log", "generate_seconds", "train_exit", "train_log",
        "train_seconds", "checkpoint", "test_exit", "test_log", "test_seconds", "status"
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print("\nSaved summary files:")
print(results_json)
print(results_csv)
print("\nResults:")
for r in results:
    print(r)

Tomograms: ['24092002', '24092009', '24092013', '24092014', '24092016', '24092017', '24092020', '24092023', '24092024', '24092025', '24092026', '24092029', '24092030', '24092031']
Coords: ['24092002', '24092009', '24092013', '24092014', '24092016', '24092017', '24092020', '24092023', '24092024', '24092025', '24092026', '24092029', '24092030', '24092031']

Tomogram index mapping:
  0: 24092002
  1: 24092009
  2: 24092013
  3: 24092014
  4: 24092016
  5: 24092017
  6: 24092020
  7: 24092023
  8: 24092024
  9: 24092025
  10: 24092026
  11: 24092029
  12: 24092030
  13: 24092031

Saved preprocessing config to /data/transformer_project/transforemer_model/test_data/dxd20240920/DeepETPicker/ribosome/ribosome_preprocess.py
[ribosome_preprocess.log] exit=0, duration=43.62s
5/14 finished.
6/14 finished.
11/14 finished.
2/14 finished.
7/14 finished.
8/14 finished.
10/14 finished.
9/14 finished.
4/14 finished.
1/14 finished.
3/14 finished.
14/14 finished.
12/14 finished.
13/14 finished.
Standardiz